# AeroPulse PM2.5 — Phase 7: Peak and Hazard Models

Build the two models an alerting product actually needs, calibrate the alarm, measure how
much warning it gives, and prove the bundle is deployable.

## Objective

This notebook replaces the Phase-6 mixture-of-experts, which must not be promoted for two
independent reasons the roadmap records:

1. **It was worse.** Mixture-of-experts RMSE 19.28 against a single model's 17.99. The
   routing added variance without adding skill.
2. **It could not ship.** The bundle was a **1.5 GB** joblib built on Random Forests —
   large enough that it was committed to git history and caused the repository bloat this
   work started from.

The replacement is three small models with clearly separated jobs:

| Model | Target | Question it answers |
|---|---|---|
| Concentration | `target_pm25_t_plus_24h` | how dirty will it be at this hour tomorrow |
| Peak | `target_max_pm25_next_24h` | how bad will it get at all |
| Hazard | `target_extreme_within_24h` | should an alert fire |

The hazard model uses **LightGBM, not Random Forest**. That is the change that turns 1.5 GB
into single-digit megabytes: a Random Forest stores every training-set leaf, so its size
scales with the data, while a boosted ensemble stores a fixed number of shallow trees.

### Why a separate classifier at all

Phase 5 tried to answer the alerting question by thresholding a regression, and got **zero
extreme recall at 24 hours**. A conditional-mean regression is trained to minimise average
error; the mean of a skewed distribution sits below the value that needs the warning. The
alarm has to be its own objective.

### What "calibrated" buys, and where the threshold comes from

A probability of 0.8 that occurs 30% of the time is unusable, because no operator can
choose a threshold from it. Platt scaling and isotonic regression are compared against the
raw scores on Brier score and expected calibration error.

**The threshold is swept on a calibration slice, never on the test block.** A threshold
chosen on test reports a number no deployment can reproduce.

In [1]:
# ============================================================================
# SETUP
# ============================================================================
from pathlib import Path
import os, json, time, gc, warnings, platform, subprocess, sys
import numpy as np
import pandas as pd
import joblib
from dotenv import find_dotenv, load_dotenv

from lightgbm import LGBMRegressor, LGBMClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression

import phase7_eval as ev

_dotenv = find_dotenv(usecwd=True)
if _dotenv:
    load_dotenv(_dotenv, override=True)
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path(os.getenv("AEROPULSE_PROJECT_ROOT", ".")).resolve()
EVENT_DIR = PROJECT_ROOT / "data" / "pm25" / "processed" / "event_aware"
EVENT_FILE = EVENT_DIR / "pm25_event_aware_features.parquet"
PHASE7 = PROJECT_ROOT / "artifacts" / "pm25" / "phase7"
OUT_DIR = PHASE7 / "peak_hazard"
OUT_DIR.mkdir(parents=True, exist_ok=True)

PRIMARY_H = int(os.getenv("PM25_PRIMARY_H", "24"))
EMBARGO_H = 48
# The test block is chosen by POLLUTION SEASON, not by taking the last slice of
# the calendar. The last 15% of this dataset is June-September 2026, which is
# monsoon - the cleanest air of the year. A hazard classifier evaluated there
# would be graded on a period containing almost no hazards, and would report a
# flattering, meaningless recall. The block with the episodes is the one that
# decides whether this model works.
TEST_WINDOW_DAYS = 90      # winter: the episode season, held out entirely
CALIB_WINDOW_DAYS = 60     # post-monsoon: also has episodes, so a threshold
                           # swept here is swept on real positives

# Same architecture as notebook 06, so a difference here is the target or the
# calibration, not the capacity.
LGBM_BASE = dict(n_estimators=400, learning_rate=0.05, num_leaves=63,
                 min_child_samples=40, subsample=0.9, subsample_freq=1,
                 colsample_bytree=0.8, random_state=42, verbose=-1, n_jobs=-1)

manifest = json.loads((EVENT_DIR / "dataset_manifest.json").read_text())
FEATURES = manifest["features"]

# Carry forward the loss configuration notebook 06 selected, rather than
# re-deciding it here. If 06 has not run, fall back to tail weighting, which is
# the roadmap's default recommendation.
EXP_REPORT = PHASE7 / "experiments" / "experiment_report.json"
if EXP_REPORT.exists():
    exp = json.loads(EXP_REPORT.read_text())
    CHOSEN = exp["best_on_primary_target"]["experiment"]
    print(f"Notebook 06 selected: {CHOSEN} "
          f"(extreme_bias {exp['best_on_primary_target']['extreme_bias']:+.1f})")
else:
    exp, CHOSEN = None, "E2_tail_weights"
    print("No experiment report found; defaulting to tail weighting.")

# The configuration is READ from notebook 06's report, not inferred from the
# experiment name. Names are labels, not specifications: "E7_h24" is a
# tail-weighted run whose name says nothing about weighting, so a name-parsing
# consumer would quietly fit an unweighted model and attribute the weighted
# model's metrics to it.
_cfg = (exp or {}).get("best_on_primary_target", {}).get("loss_config")
if _cfg:
    USE_LOG = bool(_cfg["log_target"])
    USE_WEIGHTS = bool(_cfg["tail_weights"])
    QUANTILE = _cfg.get("quantile")
    print(f"Regression config read from notebook 06's report (authoritative).")
else:
    # Only for a report predating the loss_config field.
    USE_LOG = "log" in CHOSEN
    USE_WEIGHTS = ("weight" in CHOSEN) or CHOSEN in ("E6b_v3_all_features",)
    QUANTILE = ({"E5_quantile_q10": 0.1, "E5_quantile_q50": 0.5,
                 "E5_quantile_q90": 0.9}).get(CHOSEN)
    print("WARNING: notebook 06's report has no loss_config block, so the "
          "configuration was inferred from the experiment name. Re-run "
          "notebook 06 to remove this guess.")
print(f"Regression config -> log1p={USE_LOG}, tail_weights={USE_WEIGHTS}, "
      f"quantile={QUANTILE}")
print(f"dataset v{manifest['dataset_version']} ({manifest['sha256_16']}), "
      f"{len(FEATURES)} features")

Notebook 06 selected: E2_tail_weights (extreme_bias -135.3)
Regression config read from notebook 06's report (authoritative).
Regression config -> log1p=False, tail_weights=True, quantile=None
dataset v3.0.0 (ba72d91a76c075ee), 184 features


## 1. Load and split

A single chronological split with three blocks, not cross-validation: this notebook is
building the artefact that would ship, and a shipped model is trained once on everything
up to a date.

```text
|------ fit ------| gap |- calibration -| gap |---- test ----|   |- recent holdout -|
   Feb-Sep 2025           post-monsoon          winter             pre-monsoon +
   (pre-monsoon,          Oct-Nov 2025          Dec-Feb            monsoon 2026
    monsoon)              has episodes          has episodes       (transfer check)
```

**Why the test block is winter rather than the most recent data.** The obvious split — hold
out the final 15% — puts the test block in June–September 2026, which is monsoon. Rain
scavenges particulates, so that block contains almost no extreme episodes. A hazard
classifier scored there would be measured on a period with nothing to detect, and the
resulting recall would say nothing about whether the model warns anyone about anything.
Winter is where the episodes are, so winter is what gets held out.

The cost is stated rather than hidden: the fit block is smaller, and it contains **no
winter at all**, because the dataset has only one winter and it is being used to test. The
model is therefore extrapolating into a season it has never seen. That is the honest
measurement given one year of data, and `01A` records why more years are needed.

The calibration slice is post-monsoon: unseen by the fit, chronologically before the test,
and — critically — containing real episodes, so a decision threshold swept on it is swept
against actual positives rather than against noise.

Everything after the test block is kept as a **recent holdout**. It is also unseen, and
scoring the same model there shows how much of the winter performance survives into a
different season.

In [2]:
# ============================================================================
# LOAD AND SPLIT - fit / calibration / test, chronological and purged
# ============================================================================
TARGETS_NEEDED = [f"target_pm25_t_plus_{PRIMARY_H}h", "target_max_pm25_next_24h",
                  "target_extreme_within_24h", "target_very_high_within_24h"]
META = ["timestamp_utc", "location_id", "season", "pollution_regime"]

t0 = time.time()
df = pd.read_parquet(EVENT_FILE, columns=FEATURES + TARGETS_NEEDED + META)
X = df[FEATURES].to_numpy("float32")
TS = pd.DatetimeIndex(df["timestamp_utc"])
Y = {t: df[t].to_numpy("float32") for t in TARGETS_NEEDED}
CUR = df["pm25"].to_numpy("float64")
meta = df[META].reset_index(drop=True)
del df
gc.collect()
print(f"Loaded {X.shape} in {time.time()-t0:.0f}s ({X.nbytes/1e9:.2f} GB)")

t_min, t_max = TS.min(), TS.max()
# The gap covers the forecast horizon (rows just before a cut describe times
# inside the block after it) plus a rolling-window margin.
gap = pd.Timedelta(hours=PRIMARY_H + EMBARGO_H)

# ---------------------------------------------------------------------------
# LOCATE THE EPISODE SEASON FROM THE DATA
# ---------------------------------------------------------------------------
# Rather than hardcode "December", find the window carrying the most hazard
# positives. If the seasonal calendar of Indian air quality shifts, or this runs
# on a different data window, the split follows the pollution rather than a date
# someone typed once.
#
# CALIBRATION AND TEST ARE SEARCHED AS ONE BLOCK, then cut in two. The obvious
# alternative - pick the test window first, take the calibration slice from
# whatever precedes it - puts calibration in the monsoon at a ~2% hazard base
# rate while the test block runs at ~17%. A decision threshold and a probability
# calibration fitted at 2% are simply wrong at 17%: both would be tuned to a
# regime the model is not being asked to operate in. Keeping both blocks inside
# the episode season is what makes the threshold transferable.
# ---------------------------------------------------------------------------
haz = Y["target_extreme_within_24h"]
day = TS.floor("D")
per_day = pd.Series(np.nan_to_num(haz), index=day).groupby(level=0).sum()
per_day = per_day.reindex(pd.date_range(per_day.index.min(), per_day.index.max(),
                                        freq="D", tz="UTC"), fill_value=0.0)
gap_days = int(np.ceil(gap / pd.Timedelta(days=1)))
BLOCK_DAYS = CALIB_WINDOW_DAYS + gap_days + TEST_WINDOW_DAYS
rolled = per_day.rolling(f"{BLOCK_DAYS}D").sum()

# The block must leave a usable fit period in front of it.
MIN_FIT_DAYS = 120
eligible = rolled[rolled.index >= t_min + pd.Timedelta(days=MIN_FIT_DAYS + BLOCK_DAYS)
                  + gap]
if eligible.empty:
    raise RuntimeError("dataset too short to hold out an episode season")
block_end = eligible.idxmax()                      # rolling() labels at the END
block_start = block_end - pd.Timedelta(days=BLOCK_DAYS)

cal_start = block_start
cal_end = cal_start + pd.Timedelta(days=CALIB_WINDOW_DAYS)
test_start = cal_end + gap
test_end = block_end
fit_end = cal_start - gap

fit_idx = np.flatnonzero(TS < fit_end)
cal_idx = np.flatnonzero((TS >= cal_start) & (TS < cal_end))
test_idx = np.flatnonzero((TS >= test_start) & (TS < test_end))
recent_idx = np.flatnonzero(TS >= test_end + gap)

def _describe(name, idx):
    if len(idx) == 0:
        print(f"{name:<12}: EMPTY")
        return
    h = haz[idx]
    print(f"{name:<12}: {len(idx):>9,} rows  {TS[idx].min().date()} -> "
          f"{TS[idx].max().date()}  "
          f"hazard+ {int(np.nansum(h)):>6,} ({np.nanmean(h):.2%})  "
          f"season {meta.loc[idx, 'season'].mode().iloc[0]}")

print()
for nm, ix in [("fit", fit_idx), ("calibration", cal_idx), ("test", test_idx),
               ("recent", recent_idx)]:
    _describe(nm, ix)
print(f"\npurge gap between blocks: {gap} "
      f"(horizon {PRIMARY_H}h + embargo {EMBARGO_H}h)")

assert TS[fit_idx].max() < TS[cal_idx].min(), "calibration must follow fit in time"
assert TS[cal_idx].max() < TS[test_idx].min(), "test must follow calibration"
assert TS[cal_idx].max() <= test_start - gap, "purge gap does not separate cal from test"
assert TS[fit_idx].max() <= cal_start - gap, "purge gap does not separate fit from cal"

# ---------------------------------------------------------------------------
# EPISODE-DENSITY GUARD
# ---------------------------------------------------------------------------
# The point of the season-aware split is that both the calibration and the test
# block contain events. If they do not, the threshold sweep and every hazard
# metric downstream are measuring noise, so this stops rather than reporting a
# number nobody should trust.
# ---------------------------------------------------------------------------
MIN_POSITIVES = 500
for nm, ix in [("calibration", cal_idx), ("test", test_idx)]:
    n_pos = int(np.nansum(haz[ix]))
    if n_pos < MIN_POSITIVES:
        raise RuntimeError(
            f"{nm} block holds only {n_pos:,} hazard positives (need "
            f"{MIN_POSITIVES:,}). A threshold or a recall estimated from this "
            "would be noise. Move the window or widen it.")
print(f"\nBoth calibration and test clear the {MIN_POSITIVES:,}-positive floor.")

# The threshold and the probability calibration are fitted on the calibration
# block and applied to the test block, so the two have to describe the same
# operating regime. If the base rates diverge, everything derived from the
# calibration slice is tuned for conditions the test block does not contain.
cal_rate = float(np.nanmean(haz[cal_idx]))
test_rate = float(np.nanmean(haz[test_idx]))
ratio = max(cal_rate, test_rate) / max(min(cal_rate, test_rate), 1e-9)
print(f"Hazard base rate  calibration {cal_rate:.2%} vs test {test_rate:.2%} "
      f"(ratio {ratio:.2f}x)")
MAX_RATE_RATIO = 3.0
if ratio > MAX_RATE_RATIO:
    raise RuntimeError(
        f"calibration and test base rates differ by {ratio:.1f}x, above the "
        f"{MAX_RATE_RATIO}x limit. A threshold swept at {cal_rate:.2%} does not "
        f"transfer to {test_rate:.2%}; both blocks must sit inside the same "
        "pollution regime.")
print(f"Base rates agree within {MAX_RATE_RATIO}x, so a threshold swept on the "
      "calibration block is meaningful on the test block.")

print(f"\nTest block composition:")
print(meta.loc[test_idx, "season"].value_counts().to_string())
print(f"\nExtreme rows in test (>=150 ug/m3): "
      f"{int((CUR[test_idx] >= 150).sum()):,} "
      f"({(CUR[test_idx] >= 150).mean():.2%})")
print("\nNote the fit block contains no winter, because the dataset holds one "
      "winter and it is the test block. Winter numbers below are therefore "
      "extrapolation into an unseen season, which is the honest reading and the "
      "reason 01A asks for more years.")

Loaded (1705252, 184) in 0s (1.26 GB)

fit         :   715,611 rows  2025-02-18 -> 2025-10-08  hazard+ 30,208 (4.23%)  season monsoon
calibration :   188,509 rows  2025-10-12 -> 2025-12-10  hazard+ 29,007 (15.40%)  season post_monsoon
test        :   244,542 rows  2025-12-14 -> 2026-03-13  hazard+ 30,932 (12.67%)  season winter
recent      :   527,838 rows  2026-03-17 -> 2026-09-09  hazard+ 16,027 (3.04%)  season monsoon

purge gap between blocks: 3 days 00:00:00 (horizon 24h + embargo 48h)

Both calibration and test clear the 500-positive floor.
Hazard base rate  calibration 15.40% vs test 12.67% (ratio 1.22x)
Base rates agree within 3.0x, so a threshold swept on the calibration block is meaningful on the test block.

Test block composition:
season
winter         204160
pre_monsoon     40382

Extreme rows in test (>=150 ug/m3): 6,369 (2.60%)

Note the fit block contains no winter, because the dataset holds one winter and it is the test block. Winter numbers below are therefore extrapo

## 2. Concentration and peak regressors

Same features, same loss, two different questions. The peak target is the more useful of
the two for alerting: an episode that peaks at 03:00 matters whether or not 03:00 is the
hour being scored, so predicting the 24-hour maximum sidesteps a timing error the product
does not care about.

In [3]:
# ============================================================================
# REGRESSORS - concentration at t+24h, and the 24h peak
# ============================================================================
def fit_regressor(target, idx_fit, label):
    """Fit with the loss configuration notebook 06 selected."""
    y = Y[target]
    keep = idx_fit[~np.isnan(y[idx_fit])]
    y_fit = y[keep].astype("float64")
    params = dict(LGBM_BASE)
    if QUANTILE is not None:
        params.update(objective="quantile", alpha=QUANTILE)
    fit_y = np.log1p(np.clip(y_fit, 0, None)) if USE_LOG else y_fit
    w = ev.tail_weights(y_fit) if USE_WEIGHTS else None
    t0 = time.time()
    m = LGBMRegressor(**params).fit(X[keep], fit_y, sample_weight=w)
    print(f"  {label:22} fitted on {len(keep):,} rows in {time.time()-t0:.0f}s")
    return m


def predict_regressor(model, idx):
    raw = model.predict(X[idx])
    return np.clip(np.expm1(raw) if USE_LOG else raw, 0, None)


def score_regressor(target, model, idx, label):
    y = Y[target]
    sel = idx[~np.isnan(y[idx])]
    y_true = y[sel].astype("float64")
    pred = predict_regressor(model, sel)
    base = ev.persistence(CUR[sel])
    m = ev.regression_metrics(y_true, pred)
    e = ev.event_metrics(y_true, pred)
    bm = ev.regression_metrics(y_true, base)
    be = ev.event_metrics(y_true, base)
    row = {"model": label, "target": target, "test_rows": len(sel), **m,
           "extreme_bias": e["extreme_bias"], "extreme_recall": e["extreme_recall"],
           "extreme_precision": e["extreme_precision"],
           "persistence_RMSE": bm["RMSE"],
           "persistence_extreme_recall": be["extreme_recall"],
           "persistence_extreme_bias": be["extreme_bias"],
           "skill_vs_persistence": ev.skill_score(m["RMSE"], bm["RMSE"])}
    return row, pred, y_true, sel


CONC_TARGET = f"target_pm25_t_plus_{PRIMARY_H}h"
print("Fitting regressors:")
conc_model = fit_regressor(CONC_TARGET, fit_idx, "concentration t+24h")
peak_model = fit_regressor("target_max_pm25_next_24h", fit_idx, "peak next 24h")

reg_rows = []
r, conc_pred, conc_y, conc_sel = score_regressor(CONC_TARGET, conc_model, test_idx,
                                                 "concentration_t24")
reg_rows.append(r)
r, peak_pred, peak_y, peak_sel = score_regressor("target_max_pm25_next_24h", peak_model,
                                                 test_idx, "peak_next_24h")
reg_rows.append(r)
regression_table = pd.DataFrame(reg_rows)
display(regression_table.round(3))

print("Per-regime performance - the table Phase 5's -103 ug/m3 came from.\n")
print("Concentration model at t+24h:")
display(ev.regime_metrics(conc_y, conc_pred).round(2))
print("Peak model (max over t+1..t+24):")
display(ev.regime_metrics(peak_y, peak_pred).round(2))
print("Persistence on the concentration target, same rows:")
display(ev.regime_metrics(conc_y, ev.persistence(CUR[conc_sel])).round(2))

# ---------------------------------------------------------------------------
# SEASONAL TRANSFER - the same fitted model on the recent (non-winter) holdout
# ---------------------------------------------------------------------------
# Both blocks are unseen by the fit, so this is a fair second reading. What it
# adds is a question the winter block cannot answer on its own: does the model
# hold up in a season with different physics, or is the winter score specific to
# winter? A large gap here means the single-model story is weaker than the
# headline number suggests.
# ---------------------------------------------------------------------------
if len(recent_idx):
    transfer = []
    for lbl, tgt, mdl in [("concentration_t24", CONC_TARGET, conc_model),
                          ("peak_next_24h", "target_max_pm25_next_24h", peak_model)]:
        rw, _, _, _ = score_regressor(tgt, mdl, recent_idx, lbl)
        rw["block"] = "recent_holdout"
        transfer.append(rw)
    transfer_table = pd.DataFrame(transfer)
    print("\nSame models, recent holdout (pre-monsoon + monsoon 2026):")
    display(transfer_table[["model", "test_rows", "R2", "RMSE", "extreme_bias",
                            "extreme_recall", "skill_vs_persistence"]].round(3))
    d_rmse = (transfer_table.iloc[0]["RMSE"] - regression_table.iloc[0]["RMSE"])
    print(f"\nConcentration RMSE moves {d_rmse:+.2f} ug/m3 from winter to the recent "
          "block. Absolute RMSE is expected to FALL in clean seasons simply "
          "because the values are smaller, so read the skill-vs-persistence "
          "column rather than RMSE when comparing across seasons.")
else:
    transfer_table = pd.DataFrame()
    print("\nNo recent holdout available after the test block.")

Fitting regressors:


  concentration t+24h    fitted on 650,476 rows in 18s


  peak next 24h          fitted on 714,835 rows in 20s


,model,target,test_rows,MAE,RMSE,R2,bias,extreme_bias,extreme_recall,extreme_precision,persistence_RMSE,persistence_extreme_recall,persistence_extreme_bias,skill_vs_persistence
0,concentration_t24,target_pm25_t_plus_24h,216666,20.401,30.922,0.377,9.597,-70.636,0.378,0.440,32.572,0.446,-51.930,0.051
1,peak_next_24h,target_max_pm25_next_24h,244122,39.100,59.869,0.193,22.716,-32.444,0.669,0.438,67.368,0.182,-118.562,0.111


Per-regime performance - the table Phase 5's -103 ug/m3 came from.

Concentration model at t+24h:


...

Persistence on the concentration target, same rows:


,regime,count,MAE,RMSE,bias
0,normal,44017,12.24,23.38,9.20
1,moderate,95228,13.67,23.20,4.13
2,high,49008,19.04,29.04,-2.01
3,very_high,22800,31.69,43.15,-12.25
4,extreme,5613,79.12,112.30,-52.71



Same models, recent holdout (pre-monsoon + monsoon 2026):


,model,test_rows,R2,RMSE,extreme_bias,extreme_recall,skill_vs_persistence
0,concentration_t24,468655,0.113,23.201,-192.929,0.027,0.142
1,peak_next_24h,527017,0.058,62.274,-161.706,0.356,0.030



Concentration RMSE moves -7.72 ug/m3 from winter to the recent block. Absolute RMSE is expected to FALL in clean seasons simply because the values are smaller, so read the skill-vs-persistence column rather than RMSE when comparing across seasons.


## 3. Hazard classifier and calibration

`scale_pos_weight` compensates for the class imbalance directly in the objective, which is
cheaper and less lossy than resampling — nothing is discarded and nothing is duplicated.

Three probability variants are compared. The choice is made on **expected calibration
error**, not on PR-AUC: Platt and isotonic are monotone transforms, so they cannot change
the ranking and therefore cannot change PR-AUC or ROC-AUC by much. What they change is
whether the number means what it says.

**Where the choice is made matters as much as what it is made on.** Isotonic regression can
fit its calibration data almost exactly, so comparing variants on the rows they were fitted
on would pick isotonic every time regardless of whether it generalises — and comparing them
on the test block would be choosing a model on the evaluation set. The calibration slice is
therefore split: the first 60% fits the calibrators, the remaining 40% picks the variant and
the threshold, and the test block is scored once at the end. Both calibration halves precede
the test block in time.

In [4]:
# ============================================================================
# HAZARD CLASSIFIER - LightGBM, not the Random Forest that produced 1.5 GB
# ============================================================================
HAZ = "target_extreme_within_24h"
y_all = Y[HAZ]
h_fit = fit_idx[~np.isnan(y_all[fit_idx])]
h_cal = cal_idx[~np.isnan(y_all[cal_idx])]
h_te = test_idx[~np.isnan(y_all[test_idx])]
y_fit, y_cal, y_te = (y_all[h_fit].astype("float64"),
                      y_all[h_cal].astype("float64"),
                      y_all[h_te].astype("float64"))

print(f"fit {len(y_fit):,} rows ({y_fit.mean():.2%} positive) | "
      f"cal {len(y_cal):,} ({y_cal.mean():.2%}) | "
      f"test {len(y_te):,} ({y_te.mean():.2%})")
if y_te.sum() < 100:
    print("WARNING: fewer than 100 hazard positives in test - treat every "
          "classification metric below as indicative only.")

pos_w = float((len(y_fit) - y_fit.sum()) / max(y_fit.sum(), 1))
t0 = time.time()
hazard_model = LGBMClassifier(**{**LGBM_BASE, "scale_pos_weight": pos_w})
hazard_model.fit(X[h_fit], y_fit)
print(f"Hazard classifier fitted in {time.time()-t0:.0f}s "
      f"(scale_pos_weight {pos_w:.1f})")

p_cal_raw = hazard_model.predict_proba(X[h_cal])[:, 1]
p_te_raw = hazard_model.predict_proba(X[h_te])[:, 1]

# ---------------------------------------------------------------------------
# THE CALIBRATION SLICE IS ITSELF SPLIT IN TWO
# ---------------------------------------------------------------------------
# Isotonic regression can fit a calibration slice almost exactly, so scoring a
# calibrator on the rows it was fitted on would always flatter isotonic and
# would pick it regardless of whether it generalises. Splitting the slice gives
# an honest comparison:
#   cal-fit  -> fits platt and isotonic
#   cal-pick -> chooses the variant AND sweeps the threshold, unseen by both
#   test     -> reported once, never used to choose anything
# Both halves sit chronologically before the test block, so nothing here reaches
# forward in time.
# ---------------------------------------------------------------------------
split = int(len(h_cal) * 0.6)
order_cal = np.argsort(TS[h_cal].to_numpy())        # chronological within the slice
fit_pos, pick_pos = order_cal[:split], order_cal[split:]
y_cfit, y_cpick = y_cal[fit_pos], y_cal[pick_pos]
p_cfit, p_cpick = p_cal_raw[fit_pos], p_cal_raw[pick_pos]
print(f"calibration slice split -> fit {len(fit_pos):,} rows "
      f"({y_cfit.mean():.2%} positive), pick {len(pick_pos):,} rows "
      f"({y_cpick.mean():.2%} positive)")

platt = LogisticRegression(max_iter=1000).fit(p_cfit.reshape(-1, 1), y_cfit)
isotonic = IsotonicRegression(out_of_bounds="clip").fit(p_cfit, y_cfit)
CAL_VARIANTS = {
    "raw": (lambda p: p),
    "platt": (lambda p: platt.predict_proba(p.reshape(-1, 1))[:, 1]),
    "isotonic": (lambda p: isotonic.predict(p)),
}

rows = []
for name, fn in CAL_VARIANTS.items():
    p_pick, p_te = fn(p_cpick), fn(p_te_raw)
    # Threshold swept on the pick half, applied to test. Never the other way.
    grid = np.linspace(0.05, 0.95, 19)
    f1_pick = [ev.classification_metrics(y_cpick, p_pick, t)["f1"] for t in grid]
    thr = float(grid[int(np.argmax(f1_pick))])
    rows.append({"variant": name, "chosen_threshold": thr,
                 "pick_f1_at_threshold": float(max(f1_pick)),
                 # Selection uses THIS column, measured on held-out calibration
                 # rows. The test column beside it is reported, never optimised.
                 "pick_calibration_error": ev.calibration_error(y_cpick, p_pick),
                 **ev.classification_metrics(y_te, p_te, thr),
                 "test_calibration_error": ev.calibration_error(y_te, p_te)})

calibration_table = pd.DataFrame(rows)
display(calibration_table.round(4))

base = ev.majority_class_baseline(y_te)
print(f"Majority-class baseline : recall {base['recall']:.3f}, f1 {base['f1']:.3f}, "
      f"brier {base['brier']:.4f}")
print(f"Test base rate          : {y_te.mean():.4f}  "
      f"(PR-AUC must beat this to be adding anything)")

# Selected on held-out calibration rows, NOT on test.
BEST_CAL = calibration_table.loc[calibration_table.pick_calibration_error.idxmin(),
                                 "variant"]
BEST_THR = float(calibration_table.loc[calibration_table.variant == BEST_CAL,
                                       "chosen_threshold"].iloc[0])
_pick_rank = list(calibration_table.sort_values("pick_calibration_error").variant)
_test_rank = list(calibration_table.sort_values("test_calibration_error").variant)
print(f"\nVariant ranking on held-out calibration: {_pick_rank}")
print(f"Variant ranking on test                : {_test_rank}")
if _pick_rank[0] != _test_rank[0]:
    print(f"  These disagree. '{_test_rank[0]}' scores better on test, but "
          f"selecting it would be choosing on the evaluation set. '{_pick_rank[0]}' "
          "is the honest choice and is what ships.")
p_te_best = CAL_VARIANTS[BEST_CAL](p_te_raw)
print(f"\nSelected calibration    : {BEST_CAL} at threshold {BEST_THR:.2f}")
print(f"  ECE on test {calibration_table.set_index('variant').loc[BEST_CAL,'test_calibration_error']:.4f}"
      f"  (raw was {calibration_table.set_index('variant').loc['raw','test_calibration_error']:.4f})")
print(f"  Brier {calibration_table.set_index('variant').loc[BEST_CAL,'brier']:.4f}"
      f"  (raw was {calibration_table.set_index('variant').loc['raw','brier']:.4f})")

fit 714,835 rows (4.23% positive) | cal 188,413 (15.40%) | test 244,122 (12.67%)


Hazard classifier fitted in 23s (scale_pos_weight 22.7)


calibration slice split -> fit 113,047 rows (13.88% positive), pick 75,366 rows (17.67% positive)


,variant,chosen_threshold,pick_f1_at_threshold,pick_calibration_error,threshold,precision,recall,f1,pr_auc,brier,positives,predicted_positives,base_rate,roc_auc,test_calibration_error
0,raw,0.40,0.5963,0.0490,0.40,0.3550,0.6657,0.4631,0.5228,0.1123,30932,58000,0.1267,0.8292,0.1248
1,platt,0.25,0.5968,0.0216,0.25,0.3489,0.6746,0.4600,0.5228,0.0951,30932,59804,0.1267,0.8292,0.0716
2,isotonic,0.30,0.5946,0.0236,0.30,0.3740,0.6354,0.4709,0.5118,0.0939,30932,52551,0.1267,0.8290,0.0710


Majority-class baseline : recall 0.000, f1 0.000, brier 0.1107
Test base rate          : 0.1267  (PR-AUC must beat this to be adding anything)

Variant ranking on held-out calibration: ['platt', 'isotonic', 'raw']
Variant ranking on test                : ['isotonic', 'platt', 'raw']
  These disagree. 'isotonic' scores better on test, but selecting it would be choosing on the evaluation set. 'platt' is the honest choice and is what ships.

Selected calibration    : platt at threshold 0.25
  ECE on test 0.0716  (raw was 0.1248)
  Brier 0.0951  (raw was 0.1123)


In [5]:
# ============================================================================
# RELIABILITY CURVE - does a stated probability match the observed rate?
# ============================================================================
# This is the table an operator needs before being told to act on a number. A
# well-calibrated row has predicted ~= observed; a row where predicted is 0.8
# and observed is 0.3 means the alert cries wolf at that confidence level.
# ============================================================================
def reliability(y_true, prob, n_bins=10):
    edges = np.linspace(0, 1, n_bins + 1)
    b = np.clip(np.digitize(prob, edges[1:-1]), 0, n_bins - 1)
    out = []
    for i in range(n_bins):
        m = b == i
        if m.sum():
            out.append({"bin": f"{edges[i]:.1f}-{edges[i+1]:.1f}",
                        "rows": int(m.sum()),
                        "mean_predicted": float(prob[m].mean()),
                        "observed_rate": float(y_true[m].mean()),
                        "gap": float(prob[m].mean() - y_true[m].mean())})
    return pd.DataFrame(out)


for name in ["raw", BEST_CAL] if BEST_CAL != "raw" else ["raw"]:
    print(f"Reliability - {name}:")
    display(reliability(y_te, CAL_VARIANTS[name](p_te_raw)).round(3))

# Operating curve: what an operator gets at each threshold. Precision and recall
# trade off, and a public-health alert is usually willing to buy recall with
# precision - so the choice is a policy decision, and this is the table it
# should be made from.
op = []
for thr in np.linspace(0.05, 0.95, 19):
    m = ev.classification_metrics(y_te, p_te_best, thr)
    op.append({"threshold": round(thr, 2), "precision": m["precision"],
               "recall": m["recall"], "f1": m["f1"],
               "alerts_fired": m["predicted_positives"],
               "alerts_per_1000_rows": 1000 * m["predicted_positives"] / len(y_te)})
operating = pd.DataFrame(op)
print(f"Operating curve for the {BEST_CAL} variant "
      f"(threshold chosen on calibration was {BEST_THR:.2f}):")
display(operating.round(4))

Reliability - raw:


,bin,rows,mean_predicted,observed_rate,gap
0,0.0-0.1,96337,0.039,0.025,0.014
1,0.1-0.2,41810,0.146,0.062,0.085
2,0.2-0.3,27830,0.247,0.096,0.152
3,0.3-0.4,20145,0.348,0.135,0.213
4,0.4-0.5,15382,0.448,0.175,0.273
5,0.5-0.6,11852,0.547,0.225,0.322
6,0.6-0.7,9343,0.648,0.287,0.361
7,0.7-0.8,7173,0.748,0.401,0.347
8,0.8-0.9,6532,0.851,0.542,0.308
9,0.9-1.0,7718,0.949,0.795,0.154


Reliability - platt:


,bin,rows,mean_predicted,observed_rate,gap
0,0.0-0.1,133832,0.057,0.034,0.022
1,0.1-0.2,40298,0.141,0.100,0.041
2,0.2-0.3,18021,0.246,0.152,0.094
3,0.3-0.4,11726,0.347,0.183,0.164
4,0.4-0.5,8694,0.447,0.228,0.219
5,0.5-0.6,7269,0.549,0.269,0.280
6,0.6-0.7,6559,0.649,0.353,0.296
7,0.7-0.8,6523,0.751,0.457,0.294
8,0.8-0.9,11200,0.857,0.730,0.127


Operating curve for the platt variant (threshold chosen on calibration was 0.25):


,threshold,precision,recall,f1,alerts_fired,alerts_per_1000_rows
0,0.05,0.1644,0.9596,0.2807,180586,739.7367
1,0.10,0.2387,0.8511,0.3728,110290,451.7823
2,0.15,0.2836,0.7777,0.4156,84820,347.4492
3,0.20,0.3185,0.7207,0.4418,69992,286.7091
4,0.25,0.3489,0.6746,0.4600,59804,244.9759
5,0.30,0.3763,0.6322,0.4718,51971,212.8895
6,0.35,0.4047,0.5969,0.4824,45616,186.8574
7,0.40,0.4326,0.5628,0.4892,40245,164.8561
8,0.45,0.4613,0.5296,0.4931,35511,145.4642
9,0.50,0.4890,0.4987,0.4938,31551,129.2428


## 4. Event-level lead time

Row-level recall counts hours. An operator counts **episodes**, and cares about one number:
how long before an episode starts does the alarm fire?

An alert is credited to an episode only if it fires within the 24 hours before that episode
begins — the model's own horizon. An alert 40 hours early is not foresight, it is a
different alert. Lead time is therefore bounded above by 24 hours by construction, so the
right reading of the result is the distribution, not the maximum.

In [6]:
# ============================================================================
# EVENT-LEVEL LEAD TIME - per episode, not per hour
# ============================================================================
LOOKBACK_H = 24
alerted = p_te_best >= BEST_THR
test_meta = pd.DataFrame({
    "location_id": meta.loc[h_te, "location_id"].to_numpy(),
    "ts": TS[h_te],
    "pm25": CUR[h_te],
    "alert": alerted,
}).sort_values(["location_id", "ts"]).reset_index(drop=True)

events, leads = [], []
for sid, g in test_meta.groupby("location_id", sort=False):
    g = g.reset_index(drop=True)
    is_ext = (g.pm25 >= 150).to_numpy()
    # Contiguous runs above the extreme threshold, with a gap in the data
    # terminating a run rather than bridging it.
    starts = np.flatnonzero(is_ext & ~np.concatenate([[False], is_ext[:-1]]))
    for s in starts:
        t_event = g.ts.iloc[s]
        window = g[(g.ts >= t_event - pd.Timedelta(hours=LOOKBACK_H)) & (g.ts < t_event)]
        fired = window[window.alert]
        lead = ((t_event - fired.ts.iloc[0]).total_seconds() / 3600.0
                if len(fired) else np.nan)
        events.append({"location_id": sid, "event_start": t_event,
                       "peak_pm25": float(g.pm25.iloc[s:s + LOOKBACK_H].max()),
                       "detected": bool(len(fired)),
                       "lead_time_h": lead,
                       "alert_hours_in_window": int(len(fired)),
                       "hours_available": int(len(window))})
        if len(fired):
            leads.append(lead)

event_table = pd.DataFrame(events)
if event_table.empty:
    print("No extreme episodes started inside the test block - lead time cannot be "
          "measured on this split. This is a data-window limitation, not a model "
          "result: the dataset covers one post-monsoon cycle.")
    lead_summary = {}
else:
    detected = event_table.detected.mean()
    lead_summary = {
        "events": int(len(event_table)),
        "detected": int(event_table.detected.sum()),
        "detection_rate": float(detected),
        "median_lead_time_h": float(np.median(leads)) if leads else np.nan,
        "mean_lead_time_h": float(np.mean(leads)) if leads else np.nan,
        "p25_lead_time_h": float(np.percentile(leads, 25)) if leads else np.nan,
        "max_lead_time_h": float(np.max(leads)) if leads else np.nan,
    }
    print(f"Extreme episodes starting in the test block : {lead_summary['events']}")
    print(f"Detected at least once in the prior {LOOKBACK_H}h  : "
          f"{lead_summary['detected']} ({detected:.1%})")
    if leads:
        print(f"Lead time (hours)  median {lead_summary['median_lead_time_h']:.1f}, "
              f"mean {lead_summary['mean_lead_time_h']:.1f}, "
              f"p25 {lead_summary['p25_lead_time_h']:.1f}, "
              f"max {lead_summary['max_lead_time_h']:.1f}")
        print(f"\nUpper bound is {LOOKBACK_H}h by construction (the model only looks "
              f"{LOOKBACK_H}h ahead), so the median matters more than the max.")
    display(event_table.head(12))

# False alarms, event-level: an alert with no episode in the following 24h. This
# is the cost side of the recall the threshold buys.
fa = test_meta.copy()
fa["future_extreme"] = False
for sid, g in fa.groupby("location_id", sort=False):
    idx = g.index.to_numpy()
    ts_g = g.ts.to_numpy()
    ext = (g.pm25 >= 150).to_numpy()
    hit = np.zeros(len(g), bool)
    for j in range(len(g)):
        w = (ts_g > ts_g[j]) & (ts_g <= ts_g[j] + np.timedelta64(LOOKBACK_H, "h"))
        hit[j] = ext[w].any()
    fa.loc[idx, "future_extreme"] = hit

n_alerts = int(fa.alert.sum())
false_alarms = int((fa.alert & ~fa.future_extreme).sum())
print(f"\nAlerts fired            : {n_alerts:,}")
print(f"False alarms            : {false_alarms:,} "
      f"({false_alarms/max(n_alerts,1):.1%} of alerts)")
print(f"Missed extreme hours    : "
      f"{int((~fa.alert & fa.future_extreme).sum()):,}")
print("\nA public-health alarm is normally willing to buy recall with precision, so "
      "a high false-alarm share is a policy choice made at the threshold, not "
      "automatically a defect. The operating curve above is where that choice is made.")

Extreme episodes starting in the test block : 1678
Detected at least once in the prior 24h  : 1553 (92.6%)
Lead time (hours)  median 24.0, mean 20.7, p25 22.0, max 24.0

Upper bound is 24h by construction (the model only looks 24h ahead), so the median matters more than the max.


,location_id,event_start,peak_pm25,detected,lead_time_h,alert_hours_in_window,hours_available
0,5616,2025-12-14 00:30:00+00:00,356.0,False,NaN,0,0
1,5616,2025-12-15 10:30:00+00:00,250.0,True,24.0,24,24
2,5616,2025-12-15 19:30:00+00:00,195.0,True,24.0,24,24
3,5616,2025-12-16 00:30:00+00:00,195.0,True,24.0,24,24
4,5616,2025-12-16 15:30:00+00:00,309.0,True,24.0,24,24
5,5616,2025-12-16 22:30:00+00:00,309.0,True,24.0,24,24
6,5616,2025-12-17 13:30:00+00:00,309.0,True,24.0,24,24
7,5616,2025-12-19 02:30:00+00:00,341.0,True,24.0,23,23
8,5616,2025-12-19 13:30:00+00:00,341.0,True,24.0,23,23
9,5616,2025-12-20 07:30:00+00:00,246.0,True,24.0,24,24



Alerts fired            : 59,804
False alarms            : 39,045 (65.3% of alerts)
Missed extreme hours    : 9,962

A public-health alarm is normally willing to buy recall with precision, so a high false-alarm share is a policy choice made at the threshold, not automatically a defect. The operating curve above is where that choice is made.


## 5. Bundle, size and latency

The Phase-6 bundle was 1.5 GB. That is the roadmap's §38 deployment blocker, and it is
what put a 1.5 GB blob into git history.

Four things are measured, because "it's smaller" is not a deployment argument on its own:
bundle size on disk, cold start (process boot to first prediction), warm single-row
latency, and the p95 of that latency — p95 rather than the mean, because an API's
timeout budget is set by its tail.

In [7]:
# ============================================================================
# BUNDLE - persist, then measure size and latency
# ============================================================================
# Per AGENTS.md model artifacts are pickle-based and AEROPULSE_MODEL_DIR stays
# deployment-controlled, so nothing here writes outside the artifacts tree.
#
# The feature ORDER is stored with the bundle. A model served with its columns
# permuted produces confident nonsense and no error, which is the failure mode
# this field exists to prevent.
# ============================================================================
bundle = {
    "schema_version": "1.0.0",
    "dataset_version": manifest["dataset_version"],
    "dataset_fingerprint": manifest["sha256_16"],
    "created_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "feature_names": FEATURES,
    "primary_horizon_h": PRIMARY_H,
    "regression_config": {"log1p": USE_LOG, "tail_weights": USE_WEIGHTS,
                          "quantile": QUANTILE, "selected_by": CHOSEN},
    "models": {
        "concentration": conc_model,
        "peak_24h": peak_model,
        "hazard_extreme_24h": hazard_model,
    },
    "calibration": {
        "variant": BEST_CAL,
        "threshold": BEST_THR,
        "platt": platt,
        "isotonic": isotonic,
    },
    "status": "VALIDATION",     # promotion is decided by the gate in notebook 08
    "notes": "Replaces the Phase-6 mixture-of-experts. LightGBM throughout; the "
             "1.5 GB bundle came from Random Forests, whose size scales with the "
             "training set.",
}
BUNDLE_FILE = OUT_DIR / "pm25_peak_hazard_bundle.joblib"
joblib.dump(bundle, BUNDLE_FILE, compress=3)
size_mb = BUNDLE_FILE.stat().st_size / 1e6

PHASE6_BUNDLE = PROJECT_ROOT / "artifacts" / "pm25" / "phase6" / "pm25_regime_aware_bundle.joblib"
phase6_mb = PHASE6_BUNDLE.stat().st_size / 1e6 if PHASE6_BUNDLE.exists() else 1500.0
phase6_note = "measured" if PHASE6_BUNDLE.exists() else "from the roadmap; file absent"

print(f"Phase-7 bundle : {size_mb:8.2f} MB   {BUNDLE_FILE.name}")
print(f"Phase-6 bundle : {phase6_mb:8.2f} MB   ({phase6_note})")
print(f"Reduction      : {phase6_mb/max(size_mb, 1e-9):8.0f}x smaller")
print("\nPer-model contribution:")
for name, m in bundle["models"].items():
    tmp = OUT_DIR / f"_size_probe_{name}.joblib"
    joblib.dump(m, tmp, compress=3)
    print(f"  {name:22} {tmp.stat().st_size/1e6:7.2f} MB  "
          f"{m.n_estimators} trees")
    tmp.unlink()

Phase-7 bundle :     3.44 MB   pm25_peak_hazard_bundle.joblib
Phase-6 bundle :  1639.93 MB   (measured)
Reduction      :      477x smaller

Per-model contribution:
  concentration             1.05 MB  400 trees


  peak_24h                  1.08 MB  400 trees
  hazard_extreme_24h        1.31 MB  400 trees


In [8]:
# ============================================================================
# LATENCY - cold start in a FRESH process, then warm inference
# ============================================================================
# Cold start is measured in a subprocess on purpose. Timing joblib.load() in
# this kernel would measure a warm page cache and an already-imported LightGBM,
# which is not what a cold container does.
# ============================================================================
sample_row = X[test_idx[:1]].copy()
np.save(OUT_DIR / "_latency_sample.npy", sample_row)

cold_script = "\n".join([
    "import time, json",
    "t0 = time.perf_counter()",
    "import joblib, numpy as np",
    "t_import = time.perf_counter() - t0",
    "t0 = time.perf_counter()",
    f"b = joblib.load({str(BUNDLE_FILE)!r})",
    "t_load = time.perf_counter() - t0",
    f"row = np.load({str(OUT_DIR / '_latency_sample.npy')!r})",
    "t0 = time.perf_counter()",
    "b['models']['hazard_extreme_24h'].predict_proba(row)",
    "b['models']['concentration'].predict(row)",
    "t_first = time.perf_counter() - t0",
    "print(json.dumps({'import_s': t_import, 'load_s': t_load,",
    "                  'first_predict_s': t_first,",
    "                  'cold_start_s': t_import + t_load + t_first}))",
])
proc = subprocess.run([sys.executable, "-c", cold_script],
                      capture_output=True, text=True, timeout=600)
if proc.returncode != 0:
    print("cold-start probe failed:\n", proc.stderr[-1500:])
    cold = {}
else:
    cold = json.loads(proc.stdout.strip().splitlines()[-1])
    print("Cold start (fresh process):")
    for k, v in cold.items():
        print(f"  {k:18} {v*1000:9.1f} ms")

# Warm latency: single row, which is what an API request looks like.
reps = 200
lat_single, lat_full = [], []
for _ in range(reps):
    t0 = time.perf_counter()
    hazard_model.predict_proba(sample_row)
    lat_single.append(time.perf_counter() - t0)
for _ in range(reps):
    t0 = time.perf_counter()
    hazard_model.predict_proba(sample_row)
    conc_model.predict(sample_row)
    peak_model.predict(sample_row)
    lat_full.append(time.perf_counter() - t0)

# Batch throughput: the replay/backfill path scores many rows at once, where
# per-row cost is far lower than the single-row figure.
batch = X[test_idx[:10000]]
t0 = time.perf_counter()
hazard_model.predict_proba(batch)
batch_s = time.perf_counter() - t0

latency = {
    "warm_hazard_median_ms": float(np.median(lat_single) * 1000),
    "warm_hazard_p95_ms": float(np.percentile(lat_single, 95) * 1000),
    "warm_all_three_median_ms": float(np.median(lat_full) * 1000),
    "warm_all_three_p95_ms": float(np.percentile(lat_full, 95) * 1000),
    "batch_10k_total_s": float(batch_s),
    "batch_per_row_us": float(batch_s / len(batch) * 1e6),
    "reps": reps,
    **{f"cold_{k}": v for k, v in cold.items()},
}
print(f"\nWarm inference over {reps} repetitions:")
print(f"  hazard only        median {latency['warm_hazard_median_ms']:6.2f} ms   "
      f"p95 {latency['warm_hazard_p95_ms']:6.2f} ms")
print(f"  all three models   median {latency['warm_all_three_median_ms']:6.2f} ms   "
      f"p95 {latency['warm_all_three_p95_ms']:6.2f} ms")
print(f"  batch of 10,000    {latency['batch_10k_total_s']*1000:.0f} ms total, "
      f"{latency['batch_per_row_us']:.1f} us/row")
(OUT_DIR / "_latency_sample.npy").unlink()

Cold start (fresh process):
  import_s               118.4 ms
  load_s                1153.5 ms
  first_predict_s         24.6 ms
  cold_start_s          1296.5 ms

Warm inference over 200 repetitions:
  hazard only        median   0.21 ms   p95   0.28 ms
  all three models   median   0.61 ms   p95   0.69 ms
  batch of 10,000    27 ms total, 2.7 us/row


In [9]:
# ============================================================================
# ROUND-TRIP CHECK - does the persisted bundle reproduce the notebook exactly?
# ============================================================================
# A bundle that scores differently after a save/load cycle is worse than no
# bundle: the evaluation above would describe a model that was never shipped.
#
# SAFETY OF THE joblib.load BELOW: joblib is pickle-based, so loading an
# untrusted bundle is arbitrary code execution. This load is safe because
# BUNDLE_FILE was written by the cell above in this same run, inside the
# repository's artifacts tree. AGENTS.md accepts pickle for model artifacts on
# the condition that AEROPULSE_MODEL_DIR stays deployment-controlled - a bundle
# arriving from anywhere else must not be loaded without that guarantee.
# ============================================================================
reloaded = joblib.load(BUNDLE_FILE)
probe = X[test_idx[:5000]]

checks = []
p_now = hazard_model.predict_proba(probe)[:, 1]
p_rel = reloaded["models"]["hazard_extreme_24h"].predict_proba(probe)[:, 1]
checks.append(("hazard probabilities identical", np.allclose(p_now, p_rel)))
checks.append(("concentration identical",
               np.allclose(conc_model.predict(probe),
                           reloaded["models"]["concentration"].predict(probe))))
checks.append(("peak identical",
               np.allclose(peak_model.predict(probe),
                           reloaded["models"]["peak_24h"].predict(probe))))
checks.append(("feature order preserved", reloaded["feature_names"] == FEATURES))
checks.append(("threshold preserved", reloaded["calibration"]["threshold"] == BEST_THR))
checks.append(("status is VALIDATION, not PRODUCTION",
               reloaded["status"] == "VALIDATION"))

for name, ok in checks:
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}")
if not all(ok for _, ok in checks):
    raise RuntimeError("bundle round-trip failed")
del reloaded
gc.collect()
print("\nBundle round-trips cleanly.")

  [PASS] hazard probabilities identical
  [PASS] concentration identical
  [PASS] peak identical
  [PASS] feature order preserved
  [PASS] threshold preserved
  [PASS] status is VALIDATION, not PRODUCTION

Bundle round-trips cleanly.


## 6. Artifacts

`status` stays at `VALIDATION`. Promotion is the acceptance gate's decision in notebook 08,
and per `AGENTS.md` that gate is load-bearing: a model that misses stays here rather than
having its threshold relaxed.

In [10]:
# ============================================================================
# PERSIST PEAK / HAZARD ARTIFACTS
# ============================================================================
regression_table.to_csv(OUT_DIR / "regression_test_metrics.csv", index=False)
if len(transfer_table):
    transfer_table.to_csv(OUT_DIR / "seasonal_transfer_metrics.csv", index=False)
calibration_table.to_csv(OUT_DIR / "calibration_comparison.csv", index=False)
operating.to_csv(OUT_DIR / "operating_curve.csv", index=False)
ev.regime_metrics(conc_y, conc_pred).to_csv(OUT_DIR / "concentration_regime_metrics.csv",
                                            index=False)
ev.regime_metrics(peak_y, peak_pred).to_csv(OUT_DIR / "peak_regime_metrics.csv",
                                            index=False)
if not event_table.empty:
    event_table.to_csv(OUT_DIR / "event_lead_time.csv", index=False)

report = {
    "generated_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "dataset": {"version": manifest["dataset_version"],
                "fingerprint": manifest["sha256_16"]},
    "regression_config": bundle["regression_config"],
    "split": {
        "strategy": "season-aware chronological fit / calibration / test with "
                    "purge + embargo; the test block is the densest "
                    f"{TEST_WINDOW_DAYS}-day window of hazard positives, not the "
                    "most recent slice",
        "test_block_chosen_because": "the last 15% of this dataset is monsoon, "
                                     "which holds almost no episodes; a hazard "
                                     "classifier scored there would be measured "
                                     "on a period with nothing to detect",
        "fit_rows": int(len(fit_idx)), "calibration_rows": int(len(cal_idx)),
        "test_rows": int(len(test_idx)), "recent_holdout_rows": int(len(recent_idx)),
        "test_window": [str(TS[test_idx].min()), str(TS[test_idx].max())],
        "calibration_window": [str(TS[cal_idx].min()), str(TS[cal_idx].max())],
        "purge_gap_hours": PRIMARY_H + EMBARGO_H,
        "threshold_selected_on": "calibration slice only, never test",
        "known_limitation": "the fit block contains no winter, because the "
                            "dataset holds exactly one winter and it is the test "
                            "block; winter metrics are extrapolation into an "
                            "unseen season",
    },
    "regression_metrics": regression_table.to_dict("records"),
    "seasonal_transfer": (transfer_table.to_dict("records")
                          if len(transfer_table) else []),
    "hazard": {
        "scale_pos_weight": pos_w,
        "test_base_rate": float(y_te.mean()),
        "calibration_comparison": calibration_table.to_dict("records"),
        "selected_variant": BEST_CAL,
        "selected_threshold": BEST_THR,
        "majority_class_baseline": base,
    },
    "event_lead_time": lead_summary,
    "false_alarms": {"alerts": n_alerts, "false_alarms": false_alarms,
                     "false_alarm_share": float(false_alarms / max(n_alerts, 1))},
    "deployment": {
        "bundle_file": str(BUNDLE_FILE.relative_to(PROJECT_ROOT)),
        "bundle_mb": round(size_mb, 3),
        "phase6_bundle_mb": round(phase6_mb, 1),
        "phase6_size_source": phase6_note,
        "size_reduction_x": round(phase6_mb / max(size_mb, 1e-9), 1),
        **latency,
    },
    "status": "VALIDATION",
    "promotion_note": "Promotion is decided by the section-23 acceptance gate in "
                      "notebook 08. Per AGENTS.md the gate is load-bearing: a model "
                      "that misses stays at VALIDATION and the threshold is not "
                      "relaxed.",
    "caveats": [
        "One post-monsoon cycle in the dataset, so the number of extreme episodes "
        "available for lead-time measurement is small and the estimate is wide.",
        "Lead time is capped at 24h by construction because the hazard target looks "
        "24h ahead; the median is the meaningful statistic, not the maximum.",
        "Weather features are observations at t rather than forecasts valid at t+h, "
        "so a production system with forecast weather should do better at 24h.",
    ],
}
with open(OUT_DIR / "peak_hazard_report.json", "w") as f:
    json.dump(report, f, indent=2, default=str)

print("PEAK AND HAZARD MODELS COMPLETE\n")
print(f"  concentration t+{PRIMARY_H}h : R2 "
      f"{regression_table.iloc[0].R2:+.4f}, extreme bias "
      f"{regression_table.iloc[0].extreme_bias:+.1f} ug/m3")
print(f"  peak next 24h        : R2 "
      f"{regression_table.iloc[1].R2:+.4f}, extreme bias "
      f"{regression_table.iloc[1].extreme_bias:+.1f} ug/m3")
sel = calibration_table.set_index("variant").loc[BEST_CAL]
print(f"  hazard ({BEST_CAL:8}) : recall {sel.recall:.3f}, precision "
      f"{sel.precision:.3f}, PR-AUC {sel.pr_auc:.3f}, "
      f"ECE {sel.test_calibration_error:.4f}")
if lead_summary:
    print(f"  event detection      : {lead_summary['detection_rate']:.1%} of "
          f"{lead_summary['events']} episodes, median lead "
          f"{lead_summary['median_lead_time_h']:.1f} h")
print(f"  bundle               : {size_mb:.2f} MB vs Phase-6 {phase6_mb:.0f} MB "
      f"({phase6_mb/max(size_mb,1e-9):.0f}x smaller)")
print(f"  warm p95 latency     : {latency['warm_all_three_p95_ms']:.2f} ms "
      f"for all three models")
print(f"\nWrote to {OUT_DIR}:")
for p in sorted(OUT_DIR.glob("*")):
    print(f"  {p.name:38} {p.stat().st_size/1e6:8.3f} MB")

PEAK AND HAZARD MODELS COMPLETE

  concentration t+24h : R2 +0.3773, extreme bias -70.6 ug/m3
  peak next 24h        : R2 +0.1934, extreme bias -32.4 ug/m3
  hazard (platt   ) : recall 0.675, precision 0.349, PR-AUC 0.523, ECE 0.0716
  event detection      : 92.6% of 1678 episodes, median lead 24.0 h
  bundle               : 3.44 MB vs Phase-6 1640 MB (477x smaller)
  warm p95 latency     : 0.69 ms for all three models

Wrote to /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/pm25_estimator/artifacts/pm25/phase7/peak_hazard:
  calibration_comparison.csv                0.001 MB
  concentration_regime_metrics.csv          0.000 MB
  event_lead_time.csv                       0.092 MB
  operating_curve.csv                       0.002 MB
  peak_hazard_report.json                   0.008 MB
  peak_regime_metrics.csv                   0.000 MB
  pm25_peak_hazard_bundle.joblib            3.438 MB
  regression_test_metrics.csv               0.001 MB
  seasonal_transf